In [4]:
import pymupdf4llm
from pathlib import Path
from pprint import pprint

data_dir = "../data"
chunk_size = 500
pdf_files = [f"{data_dir}/1810.04805.pdf", f"{data_dir}/2302.13971.pdf"]

all_chunks = []
for pdf_path in pdf_files:
    md_text = pymupdf4llm.to_markdown(pdf_path)
    arxiv_id = Path(pdf_path).stem
    for i in range(0, len(md_text), chunk_size):
        all_chunks.append({"text": md_text[i : i + chunk_size], "arxiv_id": arxiv_id})

print(f"Parsed {len(pdf_files)} papers, {len(all_chunks)} chunks total")

Parsed 2 papers, 319 chunks total


In [5]:
all_chunks[0]

{'text': '# **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding** \n\n**Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova** Google AI Language \n\n_{_ jacobdevlin,mingweichang,kentonl,kristout _}_ @google.com \n\n## **Abstract** \n\nWe introduce a new language representation model called **BERT** , which stands for **B** idirectional **E** ncoder **R** epresentations from **T** ransformers. Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2',
 'arxiv_id': '1810.04805'}

In [6]:
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import PointStruct

client = QdrantClient(url="http://localhost:6333")

collection_name = "arxiv_papers"
embedding_model_dimensions = 384

dense_model = "BAAI/bge-small-en"
sparse_model = "qdrant/bm25"

In [7]:
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense_vector": models.VectorParams(
                size=embedding_model_dimensions, distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "bm25_sparse_vector": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        },
    )

In [8]:
# client.delete_collection(collection_name=collection_name)

In [9]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

texts = [c["text"] for c in all_chunks]

dense_encoder = SentenceTransformer(dense_model)
dense_embeddings = dense_encoder.encode(texts, show_progress_bar=True)

bm25_encoder = SparseTextEmbedding(model_name=sparse_model)
sparse_embeddings = list(bm25_encoder.embed(texts))

print(f"Dense shape: {dense_embeddings.shape}")
print(f"Sparse vectors: {len(sparse_embeddings)}")

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 10.55it/s]


Dense shape: (319, 384)
Sparse vectors: 319


In [10]:
points = []
for idx, (chunk, dense_vec, sparse_vec) in enumerate(
    zip(all_chunks, dense_embeddings, sparse_embeddings)
):
    point = PointStruct(
        id=idx + 1,
        payload={
            "text": chunk["text"],
            "arxiv_id": chunk["arxiv_id"],
            "chunk_idx": idx,
        },
        vector={
            "dense_vector": dense_vec.tolist(),
            "bm25_sparse_vector": models.SparseVector(
                indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist()
            ),
        },
    )
    points.append(point)

client.upload_points(collection_name=collection_name, points=points, batch_size=64)
print(f"Uploaded {len(points)} chunks from {len(pdf_files)} papers")

Uploaded 319 chunks from 2 papers


In [15]:
query = "How many GPU hours did it take to train LLaMA-65B?"

dense_query_vec = dense_encoder.encode(query)
sparse_query_vec = next(bm25_encoder.embed([query]))

In [16]:
results = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(query=dense_query_vec.tolist(), using="dense_vector", limit=5),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_vec.indices.tolist(),
                values=sparse_query_vec.values.tolist(),
            ),
            using="bm25_sparse_vector",
            limit=5,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)


pprint(results.points)

[ScoredPoint(id=213, version=4, score=1.0, payload={'text': 'GPU-hours|Total power<br>consumption|Carbon emitted<br>(tCO2eq)|\n|---|---|---|---|---|---|\n|OPT-175B|A100-80GB|400W|809,472|356 MWh|137|\n|BLOOM-175B|A100-80GB|400W|1,082,880|475 MWh|183|\n|LLaMA-7B|A100-80GB|400W|82,432|36 MWh|14|\n|LLaMA-13B|A100-80GB|400W|135,168|59 MWh|23|\n|LLaMA-33B|A100-80GB|400W|530,432|233 MWh|90|\n|LLaMA-65B|A100-80GB|400W|1,022,362|449 MWh|173|\n\n\n\nTable 15: **Carbon footprint of training different models in the same data center.** We follow Wu et al. (2022) to compute carb', 'arxiv_id': '2302.13971', 'chunk_idx': 212}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=169, version=3, score=0.45, payload={'text': 'answer the question. In Table 4, we report performance on NaturalQuestions, and in Table 5, we report on TriviaQA. On both benchmarks, LLaMA-65B achieve state-of-the-arts performance in the zero-shot and few-shot settings. More importantly, the LLaMA-13B is also competi